In [1]:
# Whisper API configuration
WHISPER_MODEL = "whisper-1"
OPTIMAL_SAMPLE_RATE = 16000  # Hz
CHUNK_OVERLAP_PERCENT = 0.15  # 15% overlap

# TTS API configuration
TTS_MODEL_STANDARD = "tts-1"
TTS_MODEL_HD = "tts-1-hd"
TTS_MAX_CHARS = 4000  # Character limit per TTS request

# Voice options
VOICES = {
    "narrator": "nova",  # Formal, clear narration
    "conversational": "alloy",  # Casual, friendly
    "professional": "echo",  # Professional, authoritative
    "warm": "fable",  # Warm, engaging
}


In [ ]:
from typing import Dict, Any, Optional, BinaryIO
import json
import uuid


class MockOpenAIAudioClient:
    """Mock OpenAI Audio API client that returns pre-configured responses"""

    def __init__(self):
        self.call_count = 0

    def audio_transcriptions_create(
        self,
        model: str,
        file: BinaryIO,
        language: Optional[str] = None,
        prompt: Optional[str] = None,
        response_format: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        Mock Whisper Transcription API call

        Args:
            model: Model name (whisper-1)
            file: Audio file
            language: Optional language code
            prompt: Optional context prompt
            response_format: Response format (json, text, verbose_json, etc.)

        Returns:
            Mock API response
        """
        self.call_count += 1

        # Mock transcription response
        if response_format == "verbose_json":
            return {
                "text": "This is a sample transcription of the audio content.",
                "language": language or "en",
                "words": [
                    {"word": "This", "start": 0.0, "end": 0.5},
                    {"word": "is", "start": 0.5, "end": 0.8},
                    {"word": "a", "start": 0.8, "end": 1.0},
                    {"word": "sample", "start": 1.0, "end": 1.5}
                ]
            }
        else:
            return {
                "text": "This is a sample transcription of the audio content."
            }

    def audio_speech_create(
        self,
        model: str,
        voice: str,
        input: str
    ) -> bytes:
        """
        Mock TTS API call

        Args:
            model: TTS model (tts-1, tts-1-hd)
            voice: Voice name (alloy, echo, fable, nova, etc.)
            input: Text to synthesize

        Returns:
            Mock audio data (bytes)
        """
        self.call_count += 1

        # Return mock audio data
        return b"fake audio data for testing"


# Global mock client instance
_mock_client = MockOpenAIAudioClient()


def get_openai_client():
    """Get the mock OpenAI client"""
    return _mock_client


def reset_mock_client():
    """Reset the mock client state"""
    global _mock_client
    _mock_client = MockOpenAIAudioClient()


In [ ]:
from typing import Optional, Dict, Any, BinaryIO
from api_client import get_openai_client
from config import WHISPER_MODEL


class TranscriptionService:
    """Transcribes audio using Whisper API."""

    def __init__(self):
        """Initialize the TranscriptionService."""
        self.client = get_openai_client()

    def transcribe_audio(
        self,
        audio_file: BinaryIO,
        language: Optional[str] = None,
        context_prompt: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        Transcribe audio file with optimal parameters.
        """
        response = self.client.audio_transcriptions_create(
            model=WHISPER_MODEL,
            file=audio_file,
            language=language,
            prompt=context_prompt,
            response_format="verbose_json"
        )
        return response

    def add_context_prompt(
        self,
        domain_terms: list,
        speaker_names: Optional[list] = None
    ) -> str:
        """
        Build context prompt for domain-specific terms.
        """
        prompt_parts = []

        if domain_terms:
            terms_str = ", ".join(domain_terms)
            prompt_parts.append(f"This audio contains the following terms: {terms_str}.")

        if speaker_names:
            names_str = ", ".join(speaker_names)
            prompt_parts.append(f"Speakers in this audio: {names_str}.")

        return " ".join(prompt_parts)

In [ ]:
from pathlib import Path
from typing import List, Tuple
from pydub import AudioSegment
# from config import OPTIMAL_SAMPLE_RATE, CHUNK_OVERLAP_PERCENT


class AudioPreprocessor:
    """Preprocesses audio files for Whisper API."""

    def __init__(self):
        """Initialize the AudioPreprocessor."""
        pass

    def convert_to_mono(self, audio: AudioSegment) -> AudioSegment:
        """
        Convert audio to mono channel.
        Whisper works best with mono channel audio.
        """
        return audio.set_channels(1)

    def set_sample_rate(self, audio: AudioSegment, target_rate: int = OPTIMAL_SAMPLE_RATE) -> AudioSegment:
        """
        Set audio sample rate to target rate (default 16kHz).
        Whisper is optimized for 16kHz sample rate.
        """
        return audio.set_frame_rate(target_rate)

    def chunk_audio(
        self,
        audio: AudioSegment,
        chunk_duration_ms: int = 600000  # 10 minutes default
    ) -> List[AudioSegment]:
        """
        Chunk audio into segments with overlap.

        Should include 10-20% overlap between chunks.
        """
        audio_length = len(audio)

        if audio_length <= chunk_duration_ms:
            return [audio]

        chunks = []
        overlap_ms = int(chunk_duration_ms * CHUNK_OVERLAP_PERCENT)

        start = 0
        while start < audio_length:
            end = min(start + chunk_duration_ms, audio_length)
            chunk = audio[start:end]
            chunks.append(chunk)

            if end >= audio_length:
                break

            next_start = end - overlap_ms
            start = max(next_start, start + 1)

        return chunks

In [ ]:
from typing import List, Optional
from pathlib import Path
import io
import re
from api_client import get_openai_client
from config import TTS_MODEL_STANDARD, TTS_MODEL_HD, TTS_MAX_CHARS, VOICES


class SummaryNarrator:
    """Generates audio summaries using TTS API."""

    def __init__(self):
        """Initialize the SummaryNarrator."""
        self.client = get_openai_client()

    def select_voice(self, content_tone: str) -> str:
        """
        Select appropriate voice based on content tone.
        """
        tone_mapping = {
            "formal": "narrator",
            "conversational": "conversational",
            "professional": "professional",
            "warm": "warm"
        }

        return VOICES.get(tone_mapping.get(content_tone, "nova"), "nova")

    def chunk_text_for_tts(self, text: str, max_chars: int = TTS_MAX_CHARS) -> List[str]:
        """
        Chunk text for TTS API (4000 character limit).
        Should split at sentence boundaries, not mid-sentence.
        """
        if len(text) <= max_chars:
            return [text]

        sentences = re.split(r'([.!?]+)', text)

        chunks = []
        current_chunk = ""

        i = 0

        while i < len(sentences):
            sentence = sentences[i]

            if i + 1 < len(sentences):
                sentence += sentences[i + 1]  # Add punctuation
                i += 1

            if len(current_chunk) + len(sentence) <= max_chars:
                current_chunk += sentence
            else:
                if current_chunk:
                    chunks.append(current_chunk)
                current_chunk = sentence

            i += 1

        if current_chunk:
            chunks.append(current_chunk)  # Append any remaining text

        return chunks

    def select_tts_model(self, use_case: str) -> str:
        """
        Select appropriate TTS model based on use case.
        """
        if use_case == "quality":
            return TTS_MODEL_HD
        else:
            return TTS_MODEL_STANDARD

    def generate_audio(
        self,
        text: str,
        content_tone: str = "formal",
        use_case: str = "standard"
    ) -> bytes:
        """Generate audio from text using TTS API."""
        voice = self.select_voice(content_tone)
        model = self.select_tts_model(use_case)

        # Handle text length limits
        text_chunks = self.chunk_text_for_tts(text)

        # Generate audio for each chunk
        audio_chunks = []
        for chunk in text_chunks:
            audio = self.client.audio_speech_create(
                model=model,
                voice=voice,
                input=chunk
            )
            audio_chunks.append(audio)

        # Combine audio chunks if multiple
        if len(audio_chunks) > 1:
            # In real implementation, would combine audio files
            return audio_chunks[0]

        return audio_chunks[0] if audio_chunks else b""
